# Skill 基础教程 (v0.3)

本教程帮助你理解 Skill 的概念和使用方式。

## 学习目标

1. 理解什么是 Skill，与 MCP 的区别
2. 掌握 SKILL.md 的格式和结构
3. 学会使用 SkillManager 发现和加载 Skills
4. 理解 Skill 如何与 LLM 集成

In [ ]:
# Step 1: 安装依赖
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", 
    "mcp", "openai", "python-dotenv", "-q"
])
print("依赖安装完成！")

In [ ]:
# Step 2: 设置项目路径
import sys
from pathlib import Path

cwd = Path.cwd()
if cwd.name == "notebooks":
    project_root = cwd.parent
else:
    project_root = cwd
    while not (project_root / "pyproject.toml").exists():
        project_root = project_root.parent

src_path = project_root / "src"
sys.path.insert(0, str(src_path))

print(f"项目根目录: {project_root}")
print(f"源码路径: {src_path}")

## 1. 什么是 Skill？

Skill 是**可执行知识包**，包含：
- 元数据（name, description）
- 指令内容（Markdown 格式）
- 可选的辅助脚本和资源

### Skill 与 MCP 的区别

| 特性 | MCP | Skill |
|------|-----|-------|
| 类型 | 工具/函数 | 提示词扩展 |
| 输入 | 结构化参数 (JSON) | 自然语言 |
| 输出 | 结构化结果 | 指导性内容 |
| 调用方式 | Tool Call | 上下文注入 |
| 适合场景 | 明确的输入输出操作 | 需要灵活处理的任务 |

## 2. SKILL.md 格式

每个 Skill 是一个文件夹，包含 `SKILL.md` 文件：

```
skills/
├── calculator/
│   └── SKILL.md
├── file_reader/
│   └── SKILL.md
└── code_review/
    └── SKILL.md
```

### SKILL.md 结构

```markdown
---
name: calculator
description: 执行数学计算，支持基本运算和常用函数
---

# Calculator Skill

使用 Python 进行数学计算...
```

In [ ]:
# 查看内置 Skills 目录
from rogue_rabbit.skills import BUILTIN_SKILLS_DIR

print(f"Skills 目录: {BUILTIN_SKILLS_DIR}")
print(f"\n目录内容:")
for item in BUILTIN_SKILLS_DIR.iterdir():
    if item.is_dir():
        skill_md = item / "SKILL.md"
        status = "✅" if skill_md.exists() else "❌"
        print(f"  {status} {item.name}/")

## 3. 使用 SkillManager

SkillManager 负责：
- **发现**: 扫描目录，找到所有 SKILL.md 文件
- **加载**: 解析 SKILL.md（YAML frontmatter + Markdown）
- **管理**: 维护 skill 注册表，提供查询接口

In [ ]:
# 导入 SkillManager
from rogue_rabbit.core.skill_manager import SkillManager
from rogue_rabbit.skills import get_skill_dirs

# 获取 skill 搜索目录
skill_dirs = get_skill_dirs()
print(f"Skill 搜索目录: {skill_dirs}")

# 创建 SkillManager
manager = SkillManager(skill_dirs)

In [ ]:
# 发现所有 Skills
result = manager.discover()

print(f"发现 {len(result.skills)} 个 Skill:")
for meta in result.skills:
    print(f"  - {meta.name}: {meta.description}")

if result.errors:
    print(f"\n加载失败:")
    for error in result.errors:
        print(f"  - {error}")

In [ ]:
# 加载指定 Skill
skill = manager.load("calculator")

if skill:
    print(f"Skill 名称: {skill.meta.name}")
    print(f"Skill 描述: {skill.meta.description}")
    print(f"Skill 路径: {skill.base_path}")
    print(f"\n内容预览:")
    print(skill.content[:300] + "...")

## 4. Skill 与 LLM 集成

Skill 通过**上下文注入**方式与 LLM 集成：

1. 用户提问
2. LLM 检查可用 Skills
3. LLM 选择合适的 Skill
4. 系统注入 Skill 内容到上下文
5. LLM 根据 Skill 指导执行任务

In [ ]:
# 获取 Skill 的完整提示词（用于注入上下文）
full_prompt = skill.get_full_prompt()
print(f"完整提示词长度: {len(full_prompt)} 字符")
print(f"\n完整提示词:")
print(full_prompt)

In [ ]:
# 获取所有 Skill 描述（用于 LLM prompt）
skill_descriptions = manager.get_skill_descriptions()
print(skill_descriptions)

## 5. 创建自定义 Skill

创建自定义 Skill 很简单：

1. 在 `skills/` 目录下创建新文件夹
2. 创建 `SKILL.md` 文件
3. 添加 YAML frontmatter 和 Markdown 内容

In [ ]:
# 示例：创建一个简单的 Skill
skill_content = '''---
name: greeter
description: 生成友好的问候语
---

# Greeter Skill

根据用户的名字和时间生成问候语：

- 早上 (6-12点): "早上好，{name}！"
- 下午 (12-18点): "下午好，{name}！"
- 晚上 (18-24点): "晚上好，{name}！"

总是保持友好和热情的语气。
'''

print(skill_content)

## 总结

### 核心概念

1. **Skill**: 可执行知识包，包含元数据和指令
2. **SKILL.md**: YAML frontmatter + Markdown 格式
3. **SkillManager**: 发现、加载、管理 Skills
4. **上下文注入**: Skill 通过注入方式扩展 LLM 能力

### 与 MCP 的配合

- **Skill**: 指导 LLM "如何" 做某事
- **MCP**: 提供 "工具" 让 LLM 执行具体操作
- **组合使用**: Skill 指导 + MCP 工具执行

### 下一步

- 运行 `experiments/08_skill_basic.py` 和 `09_skill_with_llm.py`
- 创建自己的 Skill 并测试
- 尝试 Skill + MCP 组合使用